<a href="https://colab.research.google.com/github/saverin0/Change-Detection-Using-Dinov3/blob/main/99_reset_training_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Utility: Reset a Training Run

This is not a pipeline step. It's a small tool for one situation: **you want to retrain a Part 3 run under a name that already has results.**

### Why it exists
Part 3 is resumable. When `training/<RUN_NAME>/last.pt` exists, it continues that run from its last checkpoint, and if the run already finished or early-stopped, it stops immediately without training. That's the right behaviour after a Colab disconnect, but it means a finished run can't be retrained from scratch just by running Part 3 again.

In this project it was used once, after the cloud-mask fix made the first run's results invalid: the old checkpoints were archived, and Part 3 was retrained on the corrected labels under the same name.

### When you don't need it
- **Trying a new setting:** give it a new `RUN_NAME` in Part 3. Every run has its own folder and its own results, and Part 3 (Step 15) and Part 4 (Step 11) compare all of them.
- **Resuming after a disconnect:** just run Part 3 again.

### What it touches
Only these six files, inside **one run's folder**, `spacenet7_cache/training/<RUN_NAME>/`:

| File | Action |
|---|---|
| `best.pt`, `last.pt` | model checkpoints, removed |
| `history.json`, `test_metrics.json` | run results, removed |
| `training_curves.png`, `test_predictions.png` | figures, removed |

**Never touched:** `training/split.json` (shared by all runs, one folder up), other runs' folders, labels, cloud masks, features, PCA, and raw data.

`RUN_NAME = ""` targets `training/` itself, where the very first run (before run folders existed) saved its files.

### How to use
1. Set `RUN_NAME` in Step 2 and run all cells with `ACTION = "preview"` (the default). It only lists what exists.
2. Set `ACTION` to `"archive"` (recommended: moves the files into `training/<RUN_NAME>/archive_<date_time>/`, so they can be restored) or `"delete"` (permanent).
3. Run the last cell again, then run Part 3 for that run name.

A CPU runtime is enough; nothing is computed here.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Step 2: Choose what to do

`CACHE_BASE` must be the same `cache_base` used in Parts 1–3. `RUN_NAME` must match the run you want to reset.

In [ ]:
from datetime import datetime
from pathlib import Path
import shutil

CACHE_BASE = Path("/content/drive/MyDrive/datasets/spacenet7_cache")

RUN_NAME = "grid64_pca256"
ACTION = "preview"  # "preview", "archive", or "delete"

TRAINING_DIR = CACHE_BASE / "training" / RUN_NAME
FILES_TO_RESET = [
    "best.pt",
    "last.pt",
    "history.json",
    "test_metrics.json",
    "training_curves.png",
    "test_predictions.png",
]

if ACTION not in ("preview", "archive", "delete"):
    raise ValueError(f'ACTION must be "preview", "archive", or "delete", not {ACTION!r}')
if not TRAINING_DIR.is_dir():
    raise FileNotFoundError(f"{TRAINING_DIR} does not exist. Nothing to reset.")
print(f"Run folder: {TRAINING_DIR}")
print(f"Action: {ACTION}")

## Step 3: Preview, archive, or delete

This lists every file in the reset list and whether it exists. With `"archive"` or `"delete"` it then acts on the files that exist, and finally shows what's left in the run folder.

In [ ]:
existing = [TRAINING_DIR / name for name in FILES_TO_RESET if (TRAINING_DIR / name).exists()]

print("Reset list:")
for name in FILES_TO_RESET:
    path = TRAINING_DIR / name
    status = f"{path.stat().st_size / 1e6:8.2f} MB" if path.exists() else "   (not found)"
    print(f"   {name:22s} {status}")

if ACTION == "preview":
    print(f"\nPreview only: {len(existing)} file(s) would be reset. Set ACTION to \"archive\" or \"delete\" and run this cell again.")
elif not existing:
    print("\nNothing to reset.")
elif ACTION == "archive":
    archive_dir = TRAINING_DIR / f"archive_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    archive_dir.mkdir()
    for path in existing:
        shutil.move(str(path), str(archive_dir / path.name))
    print(f"\nArchived {len(existing)} file(s) to {archive_dir}")
else:
    for path in existing:
        path.unlink()
    print(f"\nDeleted {len(existing)} file(s).")

print(f"\nNow in {TRAINING_DIR.name or 'training'}/:")
for path in sorted(TRAINING_DIR.iterdir()):
    print(f"   {path.name}{'/' if path.is_dir() else ''}")